# Preference collection and judge audits

A comparison becomes training data only after its identity, rubric, presentation and outcome have been recorded correctly. Here we separate an answer from its A/B slot, distinguish a tie from a failure, and measure controlled judge vulnerabilities before any model is trained.

This lesson uses original **authored** texts/reference labels and **simulated** deterministic judge rules. No human annotation study, AI output, private label, download or live judge call is involved. High content-rule agreement is a fixture control, not evidence of evaluator quality.

Follow the cycle: predict → expose the record → change one condition → inspect the visualization → state what the evidence does not prove. The companion [chapter](../../book/chapters/10-preferences-and-reward-models.md) and [worked solutions](../../book/solutions/10-preferences-and-reward-models.md) connect the audit to Bradley–Terry learning.

In [ ]:
from pathlib import Path
import sys, json
from dataclasses import replace
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/dongxi_llms').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from dongxi_llms.preference_audit import (
    JudgeIdentity, RUBRIC, load_fixture, blind_presentation, collect_judgment,
    rubric_hash, ratings_to_pairs, ranking_to_pairs, perturb_fixture,
    simulated_collection, audit, source_pair_weights, validate_dataset,
)
plt.rcParams.update({'font.family': 'DejaVu Sans', 'figure.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False})
base_pairs, base_labels = load_fixture(ROOT / 'fixtures/preference-audit/pairs.json')
pairs, labels = perturb_fixture(base_pairs, base_labels)
records = simulated_collection(pairs)
result = audit(pairs, labels, records)
assert result['pair_count'] == 24 and result['judgment_count'] == 484
print('Authored base pairs:', len(base_pairs), '| source groups:', result['source_count'])
print('Simulated observations:', len(records), '| retained invalids:', len(result['failures']))

### Prediction

## 1 An answer is not its display slot

The first story explicitly puts the ball in a red box. Before revealing the reference: if the correct answer moves from slot A to slot B, should the canonical winner change? What information must the collector preserve so a reward learner does not accidentally receive the opposite label?

Exercise: inspect the blind display, then change `display_swapped` and `raw_slot` together. Keep the same underlying answer selected. The next cell is a complete runnable reference; you can modify its controls directly.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
pair = base_pairs[0]
judge = JudgeIdentity('lesson-manual-literal', 'authored', 'v1', RUBRIC['id'],
                      rubric_hash(), {'creation': 'authored demonstration literal'})
display_swapped = True
raw_slot = 'B'  # The red-box answer is B after the display swap.
raw = json.dumps({'verdict': raw_slot, 'reason': 'The story explicitly names this box.'})
collected = collect_judgment(pair, judge, raw, swapped=display_swapped)
print(json.dumps(blind_presentation(pair, display_swapped), indent=2))
print('Canonical outcome:', collected.outcome)
print('Selected candidate ID:', collected.selected_candidate_id)
assert collected.selected_candidate_id == pair.left.candidate_id
assert 'checkpoint_id' not in blind_presentation(pair)
# Canonical pair swapping is a different operation from presentation swapping.
assert base_labels[0].swapped().outcome == 'right'
assert pair.swapped().right.candidate_id == pair.left.candidate_id

### Reference explanation

The model-facing display contains only the rubric, question and A/B answer texts. The collector separately retains source/checkpoint identities and the mapping back to canonical candidate IDs. A B verdict after a display swap still selects the original left answer. If we instead exchange the canonical left/right candidates, the reviewed left/right label must reverse as well.

Blinding removes model names from the display; it does not guarantee neutrality. Candidate text itself may reveal style or attempt an instruction attack.


### Prediction

## 2 Make the bias interventions observable

Predict which rule fails each check: always choose A, choose the longer answer, obey an instruction inside an answer, or alternate the selection on repeated observations. Would an order audit alone detect every one of these weaknesses?

All five toy rules see the same source pairs, two display orders and two repeats. The controlled verbosity change expands only the canonical left text, with no new factual content. The instruction variant appends untrusted text only to that same answer. Authored factual reference labels remain unchanged by construction; a live study would independently review its variants.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
judge_names = ['content-rule', 'first-slot', 'longer-answer', 'injection-sensitive', 'repeat-unstable']
metric_keys = ['all_outcome_agreement', 'order_consistency', 'repeat_disagreement']
metric_labels = ['Outcome agreement ↑', 'Order consistency ↑', 'Repeat disagreement ↓',
                 'Verbosity outcome change ↓', 'Injection outcome change ↓']
values = np.array([
    [result['judges'][name][key] for key in metric_keys] +
    [result['judges'][name]['perturbations'][condition]['canonical_outcome_change_rate']
     for condition in ['verbosity', 'injection']]
    for name in judge_names
])
assert np.isfinite(values).all()
fig, ax = plt.subplots(figsize=(11, 4.4), constrained_layout=True)
image = ax.imshow(values, vmin=0, vmax=1, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(metric_labels)), metric_labels, rotation=20, ha='right')
ax.set_yticks(range(len(judge_names)), judge_names)
ax.set_title('Measured simulated judge behavior — proportions with named denominators')
for row in range(values.shape[0]):
    for col in range(values.shape[1]):
        ax.text(col, row, f'{values[row, col]:.2f}', ha='center', va='center',
                color='white' if values[row, col] > .65 else '#142538')
fig.colorbar(image, ax=ax, label='Proportion, 0 to 1')
plt.show()

### Reference explanation

![Saved reference heatmap comparing outcome agreement, order consistency, repeat disagreement and nuisance sensitivity](../figures/chapter-10/day-15-03_preference_collection_and_judges-01.png)

Read each column according to its arrow: agreement and consistency should be high; disagreement and unwanted nuisance changes should be low. A darker cell is not uniformly better. The content rule uses a transparent finite keyword list, so its perfect score is a known positive control. The first-slot rule has first-position choice rate 1 yet loses canonical consistency under order swaps. Other rules can pass the order audit while failing a nuisance or repeat audit.


### Prediction

Controlled change: replace one simulated raw verdict with a malformed string, using `collect_judgment` to construct its replacement, and rerun `audit`. Compare coverage as well as agreement. Do not edit a cached outcome alone: raw-record replay rejects that forged evidence.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
def verdict(verdict):
    return json.dumps({'verdict': verdict, 'reason': 'Explicit lesson outcome.'})
examples = [collect_judgment(pair, judge, verdict('tie'), repeat=0),
            collect_judgment(pair, judge, verdict('abstain'), repeat=1),
            collect_judgment(pair, judge, 'A is best, probably', repeat=2),
            collect_judgment(pair, judge, '', repeat=3, transport_error='Simulated collection timeout')]
for row in examples:
    print({'outcome': row.outcome, 'raw': row.raw_verdict,
           'failure_stage': row.error_stage, 'error': row.error})
assert [row.outcome for row in examples] == ['tie', 'abstain', 'invalid', 'invalid']
assert len(result['raw_judgments']) == 484 and len(result['failures']) == 4

### Reference explanation

A tie expresses equal acceptability under the rubric. An abstention expresses inability to assess. A malformed verdict or timeout is an instrument failure. Neither failure should become a half-win target. Decisive accuracy deliberately has a smaller denominator; always show nondecisive outcomes alongside it so an evaluator cannot look better merely by refusing hard comparisons.

**Exercise modification reference.** The complete reference below already constructs a malformed raw verdict with `collect_judgment(pair, judge, "A is best, probably", repeat=2)` and exposes its `invalid` outcome and parser failure. A timeout is a distinct invalid record with a transport error. For a full-panel modification, reconstruct a replacement through this collector with the original pair/judge/repeat identity, replace that observation in a copied `records` list and rerun `audit(pairs, labels, copied_records)`; never edit the derived outcome alone. That full-panel substitution remains an unexecuted extension.


In [ ]:
outcomes = ['left', 'right', 'tie', 'abstain', 'invalid']
colors = ['#356a9a', '#78a7cb', '#bebebe', '#e5b36a', '#b65d5d']
all_names = judge_names + ['malformed-controls']
fig, ax = plt.subplots(figsize=(10, 4.6), constrained_layout=True)
left = np.zeros(len(all_names))
for category, color in zip(outcomes, colors):
    counts = np.array([result['judges'][name]['outcomes'][category] for name in all_names])
    ax.barh(all_names, counts, left=left, label=category, color=color)
    left += counts
for index, total in enumerate(left):
    ax.text(total + 1, index, str(int(total)), va='center')
ax.set_xlim(0, 110)  # Keep count labels clear of the outside legend.
ax.set_xlabel('Retained observations (not independent prompts)')
ax.set_title('No malformed verdict or abstention disappears from the audit')
ax.legend(title='Canonical outcome', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.show()

![Saved reference stacked counts retaining left, right, tie, abstention and invalid verdicts](../figures/chapter-10/day-15-03_preference_collection_and_judges-02.png)

The five simulated collection rules each receive 96 observations; four failure controls are a separately identified judge. They are not mixed into its denominator as if they were actual failed model requests. Repeats and swaps contribute observations, not new independent source prompts.

### Prediction

## 3 Many rows do not mean many independent questions

The box story supplies three conversational turns; every base pair also has two nuisance variants. If every pair has equal loss weight, which source gets three times the influence of an ordinary one-turn source?

Exercise: compare row-weighted agreement with equal-source agreement. Then deliberately move one box turn to the test split. Should a source-aware audit allow it? The complete references follow.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
weights = source_pair_weights(pairs)
sources = sorted({p.source_prompt_id for p in pairs})
counts = np.array([sum(p.source_prompt_id == source for p in pairs) for source in sources])
source_totals = np.array([sum(weights[p.pair_id] for p in pairs if p.source_prompt_id == source) for source in sources])
assert np.allclose(source_totals, 1)
# One authored demonstration judge gets box turns right and abstains elsewhere.
weighted_rows = []
label_index = {label.pair_id: label for label in labels}
for p in pairs:
    if p.source_prompt_id == 'box-story':
        slot = 'A' if label_index[p.pair_id].outcome == 'left' else 'B'
    else:
        slot = 'abstain'
    weighted_rows.append(collect_judgment(p, judge, verdict(slot)))
weighted_report = audit(pairs, labels, weighted_rows)['judges'][judge.judge_id]
print('Row-weighted agreement:', weighted_report['all_outcome_agreement'])
print('Equal-source agreement:', weighted_report['source_balanced_all_outcome_agreement'])
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), constrained_layout=True)
axes[0].bar(range(len(sources)), counts, color='#356a9a')
axes[0].set_ylabel('Pair rows, including nuisance variants')
axes[0].set_title('Pair count per source')
axes[1].bar(range(len(sources)), source_totals, color='#6f937a')
axes[1].set_ylabel('Total source weight')
axes[1].set_ylim(0, 1.25)
axes[1].set_title('Source-balanced pair weighting')
for ax in axes:
    ax.set_xticks(range(len(sources)), sources, rotation=35, ha='right')
    ax.set_xlabel('Source prompt group')
plt.show()

### Reference explanation

![Saved reference pair counts compared with unit total weight for each source prompt](../figures/chapter-10/day-15-03_preference_collection_and_judges-03.png)

The box source has nine rows rather than three. Each receives weight 1/9; a one-turn source has three rows, each receiving 1/3. Equal total source weight is a declared choice, not the uniquely correct population. A traffic-weighted product objective might choose another distribution, but it must not arrive accidentally from the number of generated comparisons.

The authored demonstration has row agreement 0.5 and equal-source agreement 1/3: it gets all box turns right and correctly abstains on the unobservable question. Both metrics describe the same stored outcomes under different sampling populations.


In [ ]:
broken_pairs = list(base_pairs)
broken_pairs[1] = replace(broken_pairs[1], split='test')
try:
    validate_dataset(broken_pairs, base_labels)
except ValueError as error:
    print('Expected leakage rejection:', error)
else:
    raise AssertionError('A multi-turn source was allowed across splits')

# A new fake source ID is also insufficient: conversation siblings retain their root.
broken_pairs[1] = replace(base_pairs[1], source_prompt_id='invented-source')
try:
    validate_dataset(broken_pairs, base_labels)
except ValueError as error:
    print('Expected sibling rejection:', error)
else:
    raise AssertionError('A sibling escaped its conversation root')

### Prediction

## 4 Convert ratings and rankings without inventing certainty

Would two equal five-point ratings mean the judge cannot assess either answer? Should a score gap of four become a preference logit of four? Explain what information each conversion discards before running the reference.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
rating_pairs = ratings_to_pairs({'answer-a': 5, 'answer-b': 5, 'answer-c': 1, 'answer-d': None})
ranking_pairs = ranking_to_pairs([['answer-c'], ['answer-a', 'answer-b']],
                                  candidate_ids=['answer-a', 'answer-b', 'answer-c'])
print('Rating conversion:', json.dumps(rating_pairs, indent=2))
print('Ranking conversion:', json.dumps(ranking_pairs, indent=2))
assert rating_pairs[0]['outcome'] == 'tie'
assert rating_pairs[2]['outcome'] == 'abstain'
assert [row['outcome'] for row in ranking_pairs] == ['tie', 'right', 'right']

### Reference explanation

Observed equality becomes a tie; a missing rating produces an abstention. Ordinal scores establish an ordering but do not establish equally spaced utility or a calibrated Bradley–Terry margin. Converting a ranking of three answers produces three related pairs, not three independent prompt sources. Raw ratings/tie groups and conversion version stay beside the derived comparisons.

## What this experiment proves and what comes next

The actual reference checks the collector, strict parser, canonical slot mapping, identity/group contract and sensitivity metrics against deterministic interventions. It retains all 484 observations, including four invalid controls. It **does not** measure human reliability, live AI judge quality, general injection resistance, a trained reward model or statistically representative prompt-population performance.

For a later authorized collection, freeze the rubric and source-group split; collect responses with checkpoint/sampler identity; blind their display; retain every raw judgment and failure; review a held-out sample independently; report coverage and paired nuisance tests before constructing reward-training pairs. Use this chapter's Bradley–Terry likelihood only after deciding how ties, abstentions and invalid records enter the training policy.

[Specification](../../experiments/specs/2026-10-04-preference-audit.md) · [measured report](../../experiments/reports/2026-10-04-preference-audit.md) · [original fixture](../../fixtures/preference-audit/README.md)
